# 2a, $\mathcal{L}_{X_f}\omega = 0$

**Problem.** Let $(M, \omega)$ be a symplectic manifold with
$d\omega = 0$, and let $\pi = \omega^{-1} \in \Gamma(\wedge^2 TM)$
be the associated Poisson bivector. For $f \in C^\infty(M)$, define
the Hamiltonian vector field $X_f$ by

$$
\iota_{X_f}\omega = df, \qquad \{f, g\} := \pi(df, dg).
$$

**Goal (a).** Prove $\mathcal{L}_{X_f}\omega = 0$.

---

**Sign convention.** This notebook follows the convention of the
problem statement ($\iota_{X_f}\omega = df$). The library's default
in `HamiltonianVectorField` uses $\iota_{X_f}\omega = -df$; this
notebook does not touch that, `X_f` below is constructed as a plain
`Derivation`, and the two problem-given axioms are loaded into the
engine as inline `Definition`s.

## Strategy

Cartan's magic formula
$\mathcal{L}_X = d\circ\iota_X + \iota_X\circ d$ splits the goal
into two pieces:

$$
\mathcal{L}_{X_f}\omega = d(\iota_{X_f}\omega) + \iota_{X_f}(d\omega).
$$

Both pieces close with the given axioms:

| Piece | Axiom | Result |
|---|---|---|
| $d(\iota_{X_f}\omega)$ | $\iota_{X_f}\omega = df$ | $d(df) = 0$ (because $d^2 = 0$) |
| $\iota_{X_f}(d\omega)$ | $d\omega = 0$ (symplectic) | $\iota_{X_f}(0) = 0$ |

Sum: $0 + 0 = 0$.

Our system closes this chain in a single
`ExpandAndSimplify.prove(...)` call: the Cartan expansion comes from
`LieDerivativeCartanDefinition`, `d^2 = 0` from
`DSquaredZeroDefinition`, `ι_X(0) = 0` from the Leibniz layer, and the
two problem axioms come from two small `Definition` classes we define
in the notebook.

In [1]:
# Make jacopy importable when the notebook is opened directly.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

## 1. Setup, symbols and grading

- $f$ is a function (0-form, `Graded(degree=0)`).
- $\omega$ is a 2-form (`Graded(degree=2)`), there is no intrinsic
  p-form structure here; the system only knows it as a symbol that
  carries the right sign under graded-commutative product.
- $X_f$ is a degree-0 `Derivation` (vector field).

In [2]:
from jacopy.algebra.derivation import Act, Derivation
from jacopy.calculus.exterior_d import d, ExteriorDerivative
from jacopy.calculus.interior import interior, InteriorProduct
from jacopy.calculus.lie_derivative import lie_derivative
from jacopy.core.expr import Expr, Integer, Symbol
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry
from jacopy.proof.expansion import Definition, default_engine
from jacopy.proof.strategies import ExpandAndSimplify

reg = PropertyRegistry()

f = Symbol("f")
reg.declare(f, Graded(degree=0))

omega = Symbol("ω")
reg.declare(omega, Graded(degree=2))

X_f = Derivation("X_f", degree=0)

print("f     :", f, "  degree =", reg.get(f, Graded).degree)
print("ω     :", omega, "  degree =", reg.get(omega, Graded).degree)
print("X_f   :", X_f, "  degree =", X_f.degree)

f     : f   degree = 0
ω     : ω   degree = 2
X_f   : X_f   degree = 0


## 2. The two given axioms

The problem statement gives us two equalities, we add them as two
small `Definition`s next to the engine's built-in rules.

**Axiom 1, Symplectic closedness.** $d\omega = 0$.

**Axiom 2, Hamiltonian defining relation.** $\iota_{X_f}\omega = df$.

Each `Definition` boils down to two methods: `matches(expr)` says
whether the LHS pattern fits, and `rewrite(expr)` produces the RHS.

In [3]:
class DOmegaClosed(Definition):
    """dω = 0, the symplectic form is closed."""

    name = "dω = 0 (symplectic)"

    def matches(self, expr):
        return (
            isinstance(expr, Act)
            and isinstance(expr.op, ExteriorDerivative)
            and expr.op == d
            and expr.arg == omega
        )

    def rewrite(self, expr):
        return Integer(0)


class IotaXfOmegaIsDf(Definition):
    """ι_{X_f} ω = df, the Hamiltonian defining relation."""

    name = "ι_{X_f} ω = df"

    def matches(self, expr):
        if not isinstance(expr, Act):
            return False
        if not isinstance(expr.op, InteriorProduct):
            return False
        if expr.op.vector_field != X_f:
            return False
        return expr.arg == omega

    def rewrite(self, expr):
        return Act(d, f)


print("axiom 1:", DOmegaClosed.name)
print("axiom 2:", IotaXfOmegaIsDf.name)

axiom 1: dω = 0 (symplectic)
axiom 2: ι_{X_f} ω = df


## 3. Engine

`default_engine` brings:
- `LieDerivativeCartanDefinition`, $\mathcal{L}_X(\omega) \to (d\circ\iota_X + \iota_X\circ d)(\omega)$
- `DSquaredZeroDefinition`, $d(d(x)) \to 0$ (`d_squared_mode="axiom"`)
- `IotaOnZeroFormDefinition`, $\iota_X(0) \to 0$
- + the other built-in rules (Leibniz, Act-over-Sum, ι-square, ι(df) pairing)

We add the two problem axioms here via `register(...)`.

In [4]:
engine = default_engine(registry=reg, d_squared_mode="axiom")
engine.register(DOmegaClosed())
engine.register(IotaXfOmegaIsDf())

print(f"engine carries {len(engine.definitions)} definitions")
for defn in engine.definitions:
    print(" -", defn.name)

engine carries 10 definitions
 - L_X := d∘ι_X + ι_X∘d (Cartan definition)
 - L_X(f) = X(f) on 0-forms (flow)
 - L_X ∘ d = d ∘ L_X (flow)
 - Act linearity: (A + B)(x) = A(x) + B(x)
 - d² = 0
 - ι_X ∘ ι_X = 0
 - ι_X(f) = 0 on 0-forms
 - ι_X(df) = X(f)
 - dω = 0 (symplectic)
 - ι_{X_f} ω = df


## 4. Goal and proof

We construct LHS $= \mathcal{L}_{X_f}\omega$ and equate to
`Integer(0)`. The `ExpandAndSimplify` strategy runs the engine to a
fixed point on the obstruction $=$ LHS $-$ RHS, then applies the
simplify pipeline; if the residual drops to $0$ it returns the
chain.

In [5]:
L_Xf = lie_derivative(X_f)  # cartan-mode default
residual = Act(L_Xf, omega)

print("LHS:", residual)
print("RHS:", Integer(0))

chain = ExpandAndSimplify().prove(
    residual, Integer(0), registry=reg, engine=engine
)
print(f"\nCLOSED, {len(chain)} steps.")

LHS: L_X_f(ω)
RHS: 0

CLOSED, 7 steps.


## 5. Proof chain, LaTeX

In [6]:
from jacopy.display.jupyter import display_chain

display_chain(chain)

{\allowdisplaybreaks\scriptsize
\begin{gather*}
L_{X_f}\!\left(\omega\right) \to \left(d \, \iota_{X_f}\right)\!\left(\omega\right) + \left(\iota_{X_f} \, d\right)\!\left(\omega\right) \quad \text{[L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)]\,(axiom)} \\
\left(\left(d \, \iota_{X_f}\right)\!\left(\omega\right) + \left(\iota_{X_f} \, d\right)\!\left(\omega\right)\right) - 0 \to \left(d\!\left(\iota_{X_f}\!\left(\omega\right)\right) + \iota_{X_f}\!\left(d\!\left(\omega\right)\right)\right) - 0 \quad \text{[product-rule]}\;\text{--- graded Leibniz + linearity} \\
\iota_{X_f}\!\left(\omega\right) \to d\!\left(f\right) \quad \text{[\ensuremath{\iota}\_{X\_f} \ensuremath{\omega} = df]\,(axiom)} \\
d\!\left(d\!\left(f\right)\right) \to 0 \quad \text{[d² = 0]\,(axiom)} \\
d\!\left(\omega\right) \to 0 \quad \text{[d\ensuremath{\omega} = 0 (symplectic)]\,(axiom)} \\
\iota_{X_f}\!\left(0\right) \to 0 \quad \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)} \\
\left(0 + 0\right) - 0 \to 0 \quad \text{[simplify]}\;\text{--- canonical-form pipeline}
\end{gather*}
}

## 6. Step by step, what did each rule do?

Each step's `rule` field tells you which `Definition` fired. The
breakdown below follows the chain's flow inside the strategy:

1. **Cartan magic**, `L_Xf(ω)` → `d(ι_Xf(ω)) + ι_Xf(d(ω))` (built-in).
2. **product-rule**, the composition `d∘ι_Xf` opens, residual takes the `Sum(lhs, -rhs)` shape.
3. **ι_{X_f} ω = df**, problem axiom on the first piece.
4. **d² = 0**, `d(d(f))` is zero (built-in).
5. **dω = 0**, problem axiom on the second piece.
6. **ι_X(0) = 0**, the remaining ι drops to zero (built-in, 0-form special case).
7. **simplify**, `0 + 0 + (-0) → 0`, canonical-form closure.

In [7]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()


[1] L_X := d∘ι_X + ι_X∘d (Cartan definition) [axiom]
    L_X_f(ω)
 -> ((d * ι_X_f)(ω) + (ι_X_f * d)(ω))

[2] product-rule 
    (((d * ι_X_f)(ω) + (ι_X_f * d)(ω)) + (-0))
 -> ((d(ι_X_f(ω)) + ι_X_f(d(ω))) + (-0))

[3] ι_{X_f} ω = df [axiom]
    ι_X_f(ω)
 -> d(f)

[4] d² = 0 [axiom]
    d(d(f))
 -> 0

[5] dω = 0 (symplectic) [axiom]
    d(ω)
 -> 0

[6] ι_X(f) = 0 on 0-forms [axiom]
    ι_X_f(0)
 -> 0

[7] simplify 
    ((0 + 0) + (-0))
 -> 0



## Conclusion

$\boxed{\mathcal{L}_{X_f}\omega = 0}$ closed. The system pieces we
used:

- **Built-in**: Cartan magic expansion, Leibniz/linearity
  (`product_rule`), `d² = 0`, `ι_X(0) = 0`, the simplify pipeline.
- **Problem-specific**: two `Definition`s (`dω = 0`,
  `ι_{X_f}ω = df`). Each is 3–4 lines; loaded into the engine via
  `register(...)`.

This pattern fits almost every "close a textbook equality with two or
three given axioms" question, translating the axioms from the problem
statement into `Definition` classes is enough.